# FraudShield AI – Step 1: Data Cleaning

This notebook takes the raw Kaggle files and turns them into one clean table that is ready for modeling.

**Files needed in the `data/` folder** (from the Kaggle dataset *Transactions Fraud Datasets*):
- `transactions_data.csv`
- `cards_data.csv`
- `users_data.csv`
- `train_fraud_labels.json`  ← the fraud / not fraud answers
- `mcc_codes.json`  ← merchant category names (optional, used for readability)

> We rebuild the merge in this notebook (instead of loading your already-merged file) so that the column names are predictable and the fraud labels are included.

In [ ]:
import json
import numpy as np
import pandas as pd

DATA_DIR = "data"   # change this if your files are somewhere else
pd.set_option("display.max_columns", 50)

## 1. Load the data

The transactions file is huge (about 13 million rows), so we tell pandas to store repeated text columns as `category`. This uses much less memory.

In [ ]:
tx = pd.read_csv(
    f"{DATA_DIR}/transactions_data.csv",
    dtype={"use_chip": "category", "merchant_city": "category", "merchant_state": "category"},
)
cards = pd.read_csv(f"{DATA_DIR}/cards_data.csv")
users = pd.read_csv(f"{DATA_DIR}/users_data.csv")

print("transactions:", tx.shape)
print("cards:       ", cards.shape)
print("users:       ", users.shape)
tx.head()

## 2. Look before you clean

Always check three things first:
1. **`info()`** – what type each column is. Money columns like `amount` show up as text (`object`/`str`) because they have a `$` in them.
2. **Missing values** – which columns have blanks, and *why*.
3. **Example rows** – what the values actually look like.

In [ ]:
tx.info()

In [ ]:
tx.isna().sum()

In [ ]:
cards.head()

In [ ]:
users.head()

## 3. Add the fraud labels

The fraud answers are **not** in the CSV files. They are in `train_fraud_labels.json`, which maps each transaction id to `"Yes"` or `"No"`.

Only some transactions have a label. We can only train and test on labeled transactions, so we keep just those. This also makes the data smaller.

In [ ]:
with open(f"{DATA_DIR}/train_fraud_labels.json") as f:
    labels = json.load(f)["target"]

labels = pd.Series(labels, name="is_fraud")
labels.index = labels.index.astype("int64")          # json keys are text, ids are numbers
labels = labels.map({"Yes": 1, "No": 0}).astype("int8")

print("labeled transactions:", len(labels))
tx = tx[tx["id"].isin(labels.index)].copy()
tx["is_fraud"] = tx["id"].map(labels)
print("transactions kept:   ", len(tx))

## 4. Merge the three tables

Each table has a column called `id`, which means something different in each one. We rename them first so they do not clash:
- transactions `id` → transaction id (we keep it as `id`)
- cards `id` → `card_id`
- users `id` → `client_id`

`cards` also has a `client_id`, which is already in transactions, so we drop it from cards to avoid duplicate columns (`client_id_x`, `client_id_y`).

`validate="many_to_one"` makes pandas raise an error if a card or user appears twice, which would secretly duplicate transactions.

In [ ]:
cards = cards.rename(columns={"id": "card_id"}).drop(columns=["client_id"])
users = users.rename(columns={"id": "client_id"})

rows_before = len(tx)
df = (
    tx.merge(cards, on="card_id", how="left", validate="many_to_one")
      .merge(users, on="client_id", how="left", validate="many_to_one")
)
assert len(df) == rows_before, "Merge changed the number of rows!"
del tx  # free memory
print(df.shape)
df.head()

## 5. Remove duplicates

Each transaction should appear exactly once.

In [ ]:
print("duplicate transaction ids:", df["id"].duplicated().sum())
df = df.drop_duplicates(subset="id")

## 6. Convert money columns to numbers

Values like `"$-77.00"` or `"$24295"` are text, so the model cannot do math with them. We remove the `$` (and any commas) and convert to numbers.

In [ ]:
def money_to_float(col):
    return pd.to_numeric(col.astype(str).str.replace(r"[$,]", "", regex=True), errors="coerce").astype("float32")

money_cols = ["amount", "credit_limit", "per_capita_income", "yearly_income", "total_debt"]
for c in money_cols:
    df[c] = money_to_float(df[c])

df[money_cols].describe()

## 7. Convert dates

- `date` is the transaction date and time.
- `expires` and `acct_open_date` are written as month/year, like `12/2022`.

Once they are real dates, we can pull useful information out of them (hour, day of week, how old the account is, and so on).

In [ ]:
df["date"] = pd.to_datetime(df["date"])
df["expires"] = pd.to_datetime(df["expires"], format="%m/%Y")
df["acct_open_date"] = pd.to_datetime(df["acct_open_date"], format="%m/%Y")

df["hour"] = df["date"].dt.hour.astype("int8")
df["day_of_week"] = df["date"].dt.dayofweek.astype("int8")   # 0 = Monday
df["month"] = df["date"].dt.month.astype("int8")
df["account_age_years"] = ((df["date"] - df["acct_open_date"]).dt.days / 365.25).astype("float32")
df["years_since_pin_change"] = (df["date"].dt.year - df["year_pin_last_changed"]).astype("int16")
df["card_expired"] = (df["date"] > df["expires"]).astype("int8")

## 8. Handle missing values (and understand *why* they are missing)

Missing does not always mean "bad data":
- **`merchant_state` and `zip`** are blank for **online** purchases, because there is no physical store. We fill the state with `"ONLINE"` and add an `is_online` flag.
- **`errors`** is blank when the transaction had **no error**. That is the normal case, not missing data.

We drop `zip` because we already have the state, and thousands of zip codes would be hard for the model to use.

In [ ]:
df["is_online"] = (df["use_chip"] == "Online Transaction").astype("int8")

df["merchant_state"] = df["merchant_state"].astype(str).replace({"nan": "ONLINE"})
df.loc[df["is_online"] == 1, "merchant_state"] = "ONLINE"

# Foreign purchases: US states are 2-letter codes, other countries are written out (e.g. "Italy")
df["is_foreign"] = ((df["merchant_state"].str.len() > 2) & (df["merchant_state"] != "ONLINE")).astype("int8")

df = df.drop(columns=["zip"])

### Errors column

A transaction can have more than one error, separated by commas (for example `"Bad PIN,Insufficient Balance"`). We create:
- `has_error` – 1 if there was any error
- one 0/1 column per error type (e.g. `err_bad_pin`)

In [ ]:
df["has_error"] = df["errors"].notna().astype("int8")

err = df["errors"].astype("object").str.get_dummies(sep=",")   # blank (no error) rows become all 0s
err.columns = ["err_" + c.strip().lower().replace(" ", "_") for c in err.columns]
err = err.T.groupby(level=0).max().T.astype("int8")   # combines duplicates like "Bad PIN" and " Bad PIN"
df = pd.concat([df.drop(columns=["errors"]), err], axis=1)
print(err.sum().sort_values(ascending=False))

## 9. Refunds (negative amounts)

Negative amounts are refunds, not mistakes, so we keep them and add a flag so the model knows.

We also compare the amount to the card's credit limit. A $500 purchase is normal on a $20,000 card but unusual on a $1,000 card. Some cards have a limit of 0, so we avoid dividing by zero.

In [ ]:
df["is_refund"] = (df["amount"] < 0).astype("int8")

df["amount_to_limit"] = np.where(
    df["credit_limit"] > 0, df["amount"] / df["credit_limit"], 0
).astype("float32")

print("refunds:", df["is_refund"].sum())
print("cards with $0 limit:", (df["credit_limit"] == 0).sum())

## 10. Unusual location for this customer

The users file only has a street address, so there is no "home state" column. Instead, we find the state where each customer shops most often in person, and flag transactions that happen somewhere else.

*Note:* this looks at all of a customer's transactions, including later ones. That is fine for this project, but mention it in the report as a limitation.

In [ ]:
in_person = df[df["is_online"] == 0]
usual_state = in_person.groupby("client_id")["merchant_state"].agg(lambda s: s.value_counts().idxmax())
df["usual_state"] = df["client_id"].map(usual_state)

df["away_from_usual_state"] = (
    (df["is_online"] == 0) & (df["merchant_state"] != df["usual_state"])
).astype("int8")
df = df.drop(columns=["usual_state"])
df["away_from_usual_state"].mean()

## 11. Clean up text / yes-no columns

- `has_chip` is `"YES"`/`"NO"` → 1/0.
- Merchant category codes (`mcc`) are numbers like `5812`. We add the readable name for charts.
- Columns with only **one** value (like `card_on_dark_web`) cannot help the model, so we drop them.

In [ ]:
df["has_chip"] = (df["has_chip"].astype(str).str.upper() == "YES").astype("int8")

try:
    with open(f"{DATA_DIR}/mcc_codes.json") as f:
        mcc_names = {int(k): v for k, v in json.load(f).items()}
    df["mcc_description"] = df["mcc"].map(mcc_names).fillna("Unknown").astype("category")
except FileNotFoundError:
    print("mcc_codes.json not found – skipping category names")

constant_cols = [c for c in df.columns if df[c].nunique(dropna=False) <= 1]
print("dropping constant columns:", constant_cols)
df = df.drop(columns=constant_cols)

## 12. Drop columns that should not be used

| Column | Why we drop it |
|---|---|
| `card_number`, `cvv` | Sensitive, and just random numbers – they cannot predict fraud |
| `address` | Free text; we already have `latitude`/`longitude` |
| `birth_year`, `birth_month` | Same information as `current_age` |
| `merchant_city` | Thousands of values; `merchant_state` and our location flags cover it |
| `expires`, `acct_open_date` | Already turned into `card_expired` and `account_age_years` |

**ID columns** (`id`, `client_id`, `card_id`, `merchant_id`) are kept in the cleaned file so we can look things up, but they must be **removed before training**. Otherwise the model can memorize specific customers instead of learning fraud patterns.

In [ ]:
drop_cols = ["card_number", "cvv", "address", "birth_year", "birth_month",
             "merchant_city", "expires", "acct_open_date"]
df = df.drop(columns=[c for c in drop_cols if c in df.columns])

for c in ["use_chip", "merchant_state", "card_brand", "card_type", "gender"]:
    df[c] = df[c].astype("category")

df.info()

## 13. Final checks

1. **No missing values left.**
2. **Outliers:** look at them, but do **not** delete them. Fraud often *is* the unusual transaction, so removing outliers can remove the fraud we are trying to catch.
3. **Class balance:** fraud is extremely rare. We do **not** fix that here. Balancing (class weights or SMOTE) happens *after* the train/test split and only on the training set, otherwise the test results would be unrealistic.

In [ ]:
missing = df.isna().sum()
print("columns with missing values:\n", missing[missing > 0])

In [ ]:
df[["amount", "credit_limit", "amount_to_limit", "yearly_income", "total_debt", "credit_score"]].describe()

In [ ]:
print(df["is_fraud"].value_counts())
print(f"\nfraud rate: {df['is_fraud'].mean():.4%}")

In [ ]:
# Quick look: fraud rate by payment method and by online / foreign / error
for col in ["use_chip", "is_foreign", "has_error", "away_from_usual_state"]:
    print(df.groupby(col, observed=True)["is_fraud"].mean().rename("fraud_rate"), "\n")

## 14. Save the cleaned data

We save as **Parquet** instead of CSV. It keeps the column types (dates, categories) and is much smaller and faster to load.

We also save a small sample that keeps **all** fraud cases plus some normal ones, which is handy for quick experiments. Remember: these files go in `data/`, which is in `.gitignore`, so they will not be pushed to GitHub.

In [ ]:
df = df.sort_values("date").reset_index(drop=True)
df.to_parquet(f"{DATA_DIR}/transactions_clean.parquet", index=False)

fraud = df[df["is_fraud"] == 1]
normal = df[df["is_fraud"] == 0].sample(n=min(200_000, (df["is_fraud"] == 0).sum()), random_state=42)
sample = pd.concat([fraud, normal]).sort_values("date")
sample.to_parquet(f"{DATA_DIR}/transactions_clean_sample.parquet", index=False)

print("full:  ", df.shape)
print("sample:", sample.shape)

## Next steps (modeling notebook)

1. Load `transactions_clean.parquet`.
2. Drop the ID columns and `date`.
3. Split into train/test. Splitting by **date** (train on earlier years, test on later ones) is the most realistic, since that is how a real bank would use the model.
4. One-hot encode the category columns, and scale numbers for Logistic Regression (Random Forest does not need scaling).
5. Handle class imbalance on the **training set only**.